# Logistic Regression (cửa sổ 28 ngày)
Dự đoán `label_withdrawn` = 1 từ dữ liệu ngày 0–27. Fit trên train, chọn C và ngưỡng trên dev. Không dùng test (để tuần 5).
Phần 1: baseline (C=1, L2, ngưỡng 0,5). Phần 2 (tuần 4): tinh chỉnh C và ngưỡng.

**Số liệu train:** 19.264 dòng (lượt đăng ký) thuộc 17.394 sinh viên duy nhất; một sinh viên có thể đăng ký nhiều module nhưng chỉ thuộc một split. Dev: 4.131 dòng.

In [1]:
import sys, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.models.train_lr import main

In [2]:
metrics = main(ROOT)  # ghi 1 dòng vào results/model_metrics.csv
metrics

{'threshold': 0.5,
 'precision': 0.30186170212765956,
 'recall': 0.6045272969374168,
 'f1': 0.40266075388026606,
 'roc_auc': 0.697639439327445,
 'pr_auc': 0.3725077356198652,
 'tn': 2330,
 'fp': 1050,
 'fn': 297,
 'tp': 454}

**Nhận xét:** Recall khoảng 0,60 (bắt được 6/10 ca rút lui) nhưng Precision chỉ khoảng 0,30, nên có nhiều cảnh báo nhầm. ROC-AUC khoảng 0,70 và PR-AUC khoảng 0,37 (gấp đôi tỉ lệ nền 18,6%): baseline có tín hiệu nhưng còn yếu.

In [3]:
pd.read_csv(ROOT / 'results/model_metrics.csv')

,model,split,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,logistic_regression_C1,dev,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454
1,logistic_regression_tuned_C1,dev,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454
2,logistic_regression_C1,dev,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454


**Nhận xét:** đây là baseline để so với RF/XGBoost ở các tuần sau; ngưỡng và C sẽ được chọn ở tuần 4.

## Tinh chỉnh C và ngưỡng (trên dev)

In [4]:
from src.models.train_lr import tune_main
c_table, best_c, t_table, m, fig = tune_main(ROOT)  # ghi thêm 1 dòng vào model_metrics.csv
c_table

,C,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.01,0.5,0.298913,0.585885,0.395861,0.695741,0.362060,2348,1032,311,440
1,0.10,0.5,0.301061,0.604527,0.401948,0.697122,0.367163,2326,1054,297,454
2,1.00,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454
3,10.00,0.5,0.300000,0.603196,0.400708,0.697786,0.373765,2323,1057,298,453


**Nhận xét:** C từ 0,01 đến 10 gần như không đổi F1 (khoảng 0,40), ROC-AUC luôn khoảng 0,70. Chọn C=1 (F1 cao nhất trên dev). Mô hình ít nhạy với C vì chỉ có ít feature.

In [5]:
t_table[['threshold','precision','recall','f1']]

,threshold,precision,recall,f1
0,0.10,0.181796,1.000000,0.307661
1,0.15,0.182061,1.000000,0.308039
2,0.20,0.184627,0.994674,0.311445
3,0.25,0.193275,0.972037,0.322438
4,0.30,0.206793,0.916112,0.337420
5,0.35,0.223949,0.844208,0.353992
6,0.40,0.248595,0.765646,0.375326
7,0.45,0.273312,0.679095,0.389759
8,0.50,0.301862,0.604527,0.402661
9,0.55,0.330181,0.511318,0.401254


In [6]:
from IPython.display import Image
Image(ROOT / 'results/lr_threshold_dev.png')

**Nhận xét:** F1 cao nhất ở ngưỡng 0,5 (0,403), gần bằng 0,55 và 0,6 (0,401 và 0,400). Ngưỡng nhỏ thì Recall cao nhưng Precision thấp (ngưỡng 0,3: Recall 0,92, Precision 0,21). Mình chọn theo F1 nên ngưỡng vẫn là 0,5, kết quả tinh chỉnh giống baseline: chỉ đổi C và ngưỡng thì chưa cải thiện thêm. Nếu nhà trường muốn bắt nhiều sinh viên rút lui hơn thì có thể hạ ngưỡng xuống khoảng 0,4 (Recall 0,77, Precision 0,25), nhưng đây là lựa chọn thêm, chưa phải kết quả chính.

In [7]:
pd.read_csv(ROOT / 'results/model_metrics.csv')

,model,split,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,logistic_regression_C1,dev,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454
1,logistic_regression_tuned_C1,dev,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454
2,logistic_regression_C1,dev,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454
3,logistic_regression_tuned_C1,dev,0.5,0.301862,0.604527,0.402661,0.697639,0.372508,2330,1050,297,454
